In [1]:
import numpy as np
import pandas as pd
pdidx = pd.IndexSlice
import matplotlib
from matplotlib import pyplot as plt
from matplotlib import colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from numpy import random as rand
from scipy import *
import time as T
import sys
import json
from datetime import datetime
import os

from sklearn.metrics import roc_curve, roc_auc_score, log_loss, accuracy_score, precision_recall_curve,\
                            average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import xgboost as xgb
from xgboost import XGBRegressor
from xgboost import plot_importance
import optuna as opt

import cv2

import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# import and do some light processing to frame label data

frame_data_import = pd.read_csv('/home/travis/Projects/football_event_data_generation/camera_classifier/data_generation/frame_data/Labeled_Frame_Data_temp.csv').rename(columns={"Frame #": "Frame"})
frame_data_import = frame_data_import[["Match", "Frame", "Camera Angle"]]
frame_data_import['Frame'] = frame_data_import['Frame'].astype(int)
frame_data_import['Camera Angle'] = frame_data_import['Camera Angle'].astype(int)

#fix that one frame in the Como match
wrong_frame = (frame_data_import['Match'] == 'Como_vs_Torino') & (frame_data_import['Frame'] == 781)
frame_data_import.loc[wrong_frame, "Frame"] = 761
##

frame_data_import = frame_data_import.set_index(["Match", "Frame"]).sort_index()
master_model_data = (
    frame_data_import.groupby(level="Match", group_keys=False)
      .apply(lambda g: g.reindex(
          pd.MultiIndex.from_product(
              [[g.index.get_level_values("Match")[0]],
               range(g.index.get_level_values("Frame").min(),
                     1_000)],
              names=["Match", "Frame"]
          )
      ))
)
master_model_data["Camera Angle"] = master_model_data.groupby(level="Match")["Camera Angle"].ffill().astype(int)



In [3]:
FILE_DICT = {
    'Arsenal_vs_Manchester_City' : 'Ars_ManCit',
    'Bayern_vs_Augsburg' : 'frames_for_travis/bayern_frames_3fps',
    'Como_vs_Torino' : 'frames_for_travis/como_frames_3fps',
    'Fiorentina_vs_Cagliari' : 'Fior_Cagl',
    'Hamburg_vs_St_Pauli' : 'Ham_StPaul',
    'Legane_vs_Real_Valladolid' : 'Leg_VallD',
    'OM_vs_Lens' : 'frames_for_travis/OM_frames_3fps',
    'Sevilla_vs_Athletic' : 'frames_for_travis/sevilla_frames_30',
    'Strasbourg_vs_Auxere' : 'Stras_Aux',    
}

EXT_DICT = {
    'Arsenal_vs_Manchester_City' : 'png',
    'Bayern_vs_Augsburg' : 'jpg',
    'Como_vs_Torino' : 'jpg',
    'Fiorentina_vs_Cagliari' : 'png',
    'Hamburg_vs_St_Pauli' : 'png',
    'Legane_vs_Real_Valladolid' : 'png',
    'OM_vs_Lens' : 'jpg',
    'Sevilla_vs_Athletic' : 'jpg',
    'Strasbourg_vs_Auxere' : 'png',    

}

In [4]:
IMG_ROOT = "/home/travis/Projects/football_data_project/camera_classifier/data_generation/frame_data"
TARGET_W = 160
TARGET_H = 90

X_list = []
y_list = []
meta_list = []

matches = master_model_data.index.get_level_values("Match").unique()

for match in matches:
    subdf = master_model_data.loc[pdidx[match, :], :]
    frames = subdf.index.get_level_values("Frame")
    
    for frame in frames:
        label = master_model_data.loc[(match, frame), "Camera Angle"]
        
        img_path = os.path.join(
            IMG_ROOT,
            FILE_DICT[match],
            f"frame_{frame:06d}."+EXT_DICT[match]
        )
        
        img = cv2.imread(img_path,cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: could not read {img_path} ... ending loop")
            # continue
            break
        
        # resize to common shape
        # img_small = cv2.resize(img, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)
        img_small = cv2.resize(img, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)
        # INTRODUCE BLUR TO HELP AVOID SHORTCUTS
        img_small = cv2.GaussianBlur(img_small, (5, 5), 0) # 5 was better
        
        # flatten to 1D feature vector
        x = img_small.reshape(-1)   # same as flatten()
        
        X_list.append(x)
        y_list.append(int(label))
        meta_list.append((match, frame))

X = np.vstack(X_list)
y = np.array(y_list, dtype=np.int64)
meta = pd.DataFrame(meta_list, columns=["Match", "Frame"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print(meta.head())

X shape: (9000, 14400)
y shape: (9000,)
                        Match  Frame
0  Arsenal_vs_Manchester_City      0
1  Arsenal_vs_Manchester_City      1
2  Arsenal_vs_Manchester_City      2
3  Arsenal_vs_Manchester_City      3
4  Arsenal_vs_Manchester_City      4


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

clf.fit(X_train, y_train,verbose=True)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9605555555555556
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      1221
           1       0.99      0.88      0.94       579

    accuracy                           0.96      1800
   macro avg       0.97      0.94      0.95      1800
weighted avg       0.96      0.96      0.96      1800



## Pytorch imagle classifier example modified for football data

In [11]:
class Net_gray(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,16,5)

        self.fc1 = nn.Linear(11248,120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84,2)   # binary classifier

    def forward(self,x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x,1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [12]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

test_matches = ['Arsenal_vs_Manchester_City', 'Como_vs_Torino']

test_mask = meta[meta['Match'].isin(test_matches)].index
train_mask = meta[~meta['Match'].isin(test_matches)].index
X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

X_train = X_train.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0
y_train = y_train.astype(np.int64)
y_test  = y_test.astype(np.int64)

# grayscale: add channel dim
X_train = np.expand_dims(X_train, axis=1)   # (N,1,H,W)
X_test  = np.expand_dims(X_test, axis=1)

X_train = X_train.reshape(-1, 1, 90, 160)
X_test = X_test.reshape(-1, 1, 90, 160)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

trainset = TensorDataset(X_train_t, y_train_t)
testset  = TensorDataset(X_test_t, y_test_t)

trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
testloader  = DataLoader(testset, batch_size=32, shuffle=False)

In [13]:
device = "cpu"#torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = Net_gray().to(device)
criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

for epoch in range(10):
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, loss = {running_loss / len(trainloader):.4f}")

Epoch 1, loss = 0.2483
Epoch 2, loss = 0.1087
Epoch 3, loss = 0.0608
Epoch 4, loss = 0.0351
Epoch 5, loss = 0.0255
Epoch 6, loss = 0.0308
Epoch 7, loss = 0.0106
Epoch 8, loss = 0.0108
Epoch 9, loss = 0.0187
Epoch 10, loss = 0.0098


In [14]:
net.eval()

y_pred = []
y_true = []

with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = net(inputs)
        preds = torch.argmax(outputs, dim=1)

        y_pred.extend(preds.cpu().numpy().ravel())
        y_true.extend(labels.cpu().numpy().ravel())

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.5885
              precision    recall  f1-score   support

           0       0.91      0.45      0.60      1379
           1       0.42      0.90      0.58       621

    accuracy                           0.59      2000
   macro avg       0.67      0.68      0.59      2000
weighted avg       0.76      0.59      0.59      2000



In [65]:
meta_test = meta.iloc[test_mask].reset_index(drop=True)
wrong = np.where(np.array(y_pred) != np.array(y_true))[0]
meta_test.iloc[wrong][meta_test['Match']=='Como_vs_Torino']['Frame'].values.size

/tmp/ipykernel_683973/215051352.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  meta_test.iloc[wrong][meta_test['Match']=='Como_vs_Torino']['Frame'].values.size


89

In [66]:
test_loss = 0

with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

In [67]:
print("test loss:", test_loss / len(testloader))

test loss: 8.761745972943968


In [68]:
print("true counts:", np.bincount(np.array(y_true)))
print("pred counts:", np.bincount(np.array(y_pred)))

true counts: [1379  621]
pred counts: [ 615 1385]


In [69]:
y_pred_flip = 1 - np.array(y_pred)

print("Flipped accuracy:", accuracy_score(y_true, y_pred_flip))
print(classification_report(y_true, y_pred_flip))

Flipped accuracy: 0.453
              precision    recall  f1-score   support

           0       0.60      0.61      0.60      1379
           1       0.12      0.11      0.11       621

    accuracy                           0.45      2000
   macro avg       0.36      0.36      0.36      2000
weighted avg       0.45      0.45      0.45      2000



In [70]:
meta_test = meta.iloc[test_mask].reset_index(drop=True)

print(X_test.shape, y_test.shape, meta_test.shape)

for i in range(10):
    print(i, meta_test.iloc[i].to_dict(), y_test[i])

(2000, 1, 90, 160) (2000,) (2000, 2)
0 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 0} 0
1 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 1} 0
2 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 2} 0
3 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 3} 0
4 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 4} 0
5 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 5} 0
6 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 6} 0
7 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 7} 0
8 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 8} 0
9 {'Match': 'Arsenal_vs_Manchester_City', 'Frame': 9} 0


In [71]:
for i in range(10):
    match = meta_test.iloc[i]["Match"]
    frame = meta_test.iloc[i]["Frame"]
    label_from_meta = y_test[i]
    label_from_source = master_model_data.loc[(match, frame), "Camera Angle"]
    print(i, match, frame, label_from_meta, label_from_source)

0 Arsenal_vs_Manchester_City 0 0 0
1 Arsenal_vs_Manchester_City 1 0 0
2 Arsenal_vs_Manchester_City 2 0 0
3 Arsenal_vs_Manchester_City 3 0 0
4 Arsenal_vs_Manchester_City 4 0 0
5 Arsenal_vs_Manchester_City 5 0 0
6 Arsenal_vs_Manchester_City 6 0 0
7 Arsenal_vs_Manchester_City 7 0 0
8 Arsenal_vs_Manchester_City 8 0 0
9 Arsenal_vs_Manchester_City 9 0 0


### So clearly the small CNN is winning here. Let's try adding back color and see if the gains any advantages. Also, we will withold training data on matches to see if it makes a difference

In [9]:
IMG_ROOT = "/home/travis/Projects/football_data_project/camera_classifier/data_generation/frame_data"

ASPECT_RATIO_FACTOR = 2
TARGET_W = 160*ASPECT_RATIO_FACTOR
TARGET_H = 90*ASPECT_RATIO_FACTOR

X_list = []
y_list = []
meta_list = []

matches = master_model_data.index.get_level_values("Match").unique()

for match in matches:
    subdf = master_model_data.loc[pdidx[match, :], :]
    frames = subdf.index.get_level_values("Frame")
    
    for frame in frames:
        label = master_model_data.loc[(match, frame), "Camera Angle"]
        
        img_path = os.path.join(
            IMG_ROOT,
            FILE_DICT[match],
            f"frame_{frame:06d}."+EXT_DICT[match]
        )
        
        img = cv2.imread(img_path,cv2.COLOR_BGR2RGB)
        if img is None:
            print(f"Warning: could not read {img_path} ... ending loop")
            # continue
            break
        
        # resize to common shape
        img_small = cv2.resize(img, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)
        
        # flatten to 1D feature vector
        x = img_small.reshape(-1)   # same as flatten()
        
        X_list.append(x)
        y_list.append(int(label))
        meta_list.append((match, frame))

X = np.vstack(X_list)
y = np.array(y_list, dtype=np.int64)
meta = pd.DataFrame(meta_list, columns=["Match", "Frame"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print(meta.head())

X shape: (9000, 172800)
y shape: (9000,)
                        Match  Frame
0  Arsenal_vs_Manchester_City      0
1  Arsenal_vs_Manchester_City      1
2  Arsenal_vs_Manchester_City      2
3  Arsenal_vs_Manchester_City      3
4  Arsenal_vs_Manchester_City      4


In [ ]:
class Net_color(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        
        self.fc1 = nn.Linear(51744, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 2)

    def forward(self,x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x,1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0
y_train = y_train.astype(np.int64)
y_test  = y_test.astype(np.int64)

# grayscale: add channel dim
X_train = np.expand_dims(X_train, axis=1)   # (N,1,H,W)
X_test  = np.expand_dims(X_test, axis=1)

X_train = X_train.reshape(-1, 1, 90, 160)
X_test = X_test.reshape(-1, 1, 90, 160)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

trainset = TensorDataset(X_train_t, y_train_t)
testset  = TensorDataset(X_test_t, y_test_t)

trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
testloader  = DataLoader(testset, batch_size=32, shuffle=False)